In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Qwen2_5_SQL_Model_3"

In [ ]:
# Tạo thư mục nếu chưa tồn tại
import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Đã thiết lập thư mục lưu trên Google Drive: {DRIVE_OUTPUT_DIR}")

Đã thiết lập thư mục lưu trên Google Drive: /content/drive/MyDrive/Qwen2_5_SQL_Model_3


In [ ]:
# PHẦN 1: GIẢI NÉN DỮ LIỆU
import zipfile
import os

# Đường dẫn tới file ZIP
zip_path = "/content/database.zip"
extract_path = "/content/"  # Giải nén ngay tại thư mục hiện tại

# Giải nén file ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Đã giải nén xong! Các tệp đã được lưu tại:", extract_path)

# Kiểm tra danh sách file sau khi giải nén
print("Danh sách các tệp đã giải nén:")
for file in os.listdir(extract_path):
    print(file)

Đã giải nén xong! Các tệp đã được lưu tại: /content/
Danh sách các tệp đã giải nén:
.config
train_filtered.csv
drive
database
database.zip
valid_filtered.csv
sample_data


In [ ]:
# PHẦN 2: CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab and Kaggle notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# PHẦN 3: ĐỌC DỮ LIỆU VÀ TẠO THƯ VIỆN ÁNH XẠ
import pandas as pd
import torch
import numpy as np
import json
from datasets import Dataset
import sqlparse

# Đọc dữ liệu CSV một lần duy nhất
train_df = pd.read_csv("train_filtered.csv")
valid_df = pd.read_csv("valid_filtered.csv")

print(f"Số lượng mẫu huấn luyện: {len(train_df)}")
print(f"Số lượng mẫu kiểm thử: {len(valid_df)}")

# Kiểm tra cấu trúc dữ liệu
print("Cấu trúc dữ liệu huấn luyện:")
print(train_df.head(2))
print("Các cột có trong dữ liệu:", train_df.columns.tolist())

# Tạo ánh xạ từ db_id đến đường dẫn schema.sql
database_dir = "database"  # đường dẫn tới folder "database"
dbid2schema_path = {}

# Lấy danh sách các folder con trong database/
try:
    db_folders = [d for d in os.listdir(database_dir)
                if os.path.isdir(os.path.join(database_dir, d))]

    for db_id in db_folders:
        schema_file_path = os.path.join(database_dir, db_id, "schema.sql")
        # Kiểm tra xem file schema.sql có tồn tại không
        if os.path.exists(schema_file_path):
            dbid2schema_path[db_id] = schema_file_path
        else:
            print(f"[Warning] Không tìm thấy schema.sql trong folder {db_id}")

    print("Số lượng db_id có schema.sql:", len(dbid2schema_path))
    print("Ví dụ 5 cặp db_id -> schema_path:")
    for idx, (db_id, path) in enumerate(dbid2schema_path.items()):
        if idx >= 5:
            break
        print(f"{db_id} -> {path}")
except Exception as e:
    print(f"Lỗi khi đọc thư mục database: {e}")

# Kiểm tra một số mẫu dữ liệu kết hợp với schema
for idx, row in train_df.head(2).iterrows():
    question = row["question"]
    query = row["query"]  # Đúng với cấu trúc dữ liệu (không phải "sql")
    db_id = row["db_id"]

    # Tìm file schema.sql tương ứng
    schema_path = dbid2schema_path.get(db_id, None)
    if schema_path is None:
        print(f"[Warning] Không tìm thấy schema cho db_id={db_id}")
        continue

    try:
        # Đọc nội dung schema.sql
        with open(schema_path, "r", encoding="utf-8") as f:
            schema_content = f.read()

        # In mẫu dữ liệu
        print(f"Sample {idx} | question={question}")
        print(f"db_id={db_id} => schema_path={schema_path}")
        print(f"Query: {query}")
        print(f"Nội dung schema (vài ký tự đầu): {schema_content[:100]}")
        print("-"*40)
    except Exception as e:
        print(f"Lỗi khi đọc schema {schema_path}: {e}")

Số lượng mẫu huấn luyện: 6654
Số lượng mẫu kiểm thử: 807
Cấu trúc dữ liệu huấn luyện:
                   db_id                                              query  \
0  department_management         SELECT count(*) FROM head WHERE age  >  56   
1  department_management  SELECT name ,  born_state ,  age FROM head ORD...   

                                            question  \
0  How many heads of the departments are older th...   
1  List the name, born state and age of the heads...   

                                          query_toks  \
0  ['SELECT', 'count', '(', '*', ')', 'FROM', 'he...   
1  ['SELECT', 'name', ',', 'born_state', ',', 'ag...   

                                 query_toks_no_value  \
0  ['select', 'count', '(', '*', ')', 'from', 'he...   
1  ['select', 'name', ',', 'born_state', ',', 'ag...   

                                       question_toks  
0  ['How', 'many', 'heads', 'of', 'the', 'departm...  
1  ['List', 'the', 'name', ',', 'born', 'state', ...  
Các 

In [ ]:
# PHẦN 4: TẠO HÀM CHUẨN BỊ PROMPT
def read_schema_content(db_id):
    """Đọc nội dung schema.sql từ db_id"""
    schema_path = dbid2schema_path.get(db_id)
    if not schema_path or not os.path.exists(schema_path):
        return "Schema không tồn tại"

    try:
        with open(schema_path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"Lỗi khi đọc schema {schema_path}: {e}")
        return "Lỗi khi đọc schema"

def make_prompt(question, schema_content):
    """
    Tạo prompt theo cấu trúc mẫu của Qwen2.5 Coder
    """
    # Format đẹp cho schema SQL
    formatted_schema = sqlparse.format(
        schema_content,
        reindent=True,
        keyword_case='upper'
    )

    prompt = f"""### SQLite Schema:
```sql
{formatted_schema}
```

### Question:
{question}

### SQL Query:
"""
    return prompt

In [ ]:
# PHẦN 5: CHUẨN BỊ DỮ LIỆU CHO HUẤN LUYỆN
def prepare_data(df):
    """Chuẩn bị dữ liệu cho huấn luyện"""
    data = []
    for idx, row in df.iterrows():
        try:
            db_id = row["db_id"]
            question = row["question"]
            query = row["query"]  # Đúng với cấu trúc dữ liệu

            # Đọc schema
            schema_content = read_schema_content(db_id)
            if schema_content == "Schema không tồn tại" or schema_content == "Lỗi khi đọc schema":
                continue  # Bỏ qua mẫu này

            # Tạo prompt
            prompt = make_prompt(question, schema_content)

            # Format query đẹp hơn
            formatted_query = sqlparse.format(query, reindent=True, keyword_case='upper')

            # Thêm vào danh sách dữ liệu
            data.append({
                "instruction": "Convert the natural language question to a SQL query based on the given database schema.",
                "input": prompt,
                "output": f"```sql\n{formatted_query}\n```"
            })
        except Exception as e:
            print(f"Lỗi khi xử lý mẫu {idx}: {e}")

    return data

# Chuẩn bị dữ liệu
print("Đang chuẩn bị dữ liệu huấn luyện...")
train_data = prepare_data(train_df)
print("Đang chuẩn bị dữ liệu kiểm thử...")
valid_data = prepare_data(valid_df)

print(f"Số lượng mẫu huấn luyện sau khi xử lý: {len(train_data)}")
print(f"Số lượng mẫu kiểm thử sau khi xử lý: {len(valid_data)}")

# In ví dụ mẫu dữ liệu đã chuẩn bị
print("\nVí dụ mẫu dữ liệu đã chuẩn bị:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

# Chuyển đổi thành dataset Hugging Face
train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)

Đang chuẩn bị dữ liệu huấn luyện...
Đang chuẩn bị dữ liệu kiểm thử...
Số lượng mẫu huấn luyện sau khi xử lý: 6654
Số lượng mẫu kiểm thử sau khi xử lý: 807

Ví dụ mẫu dữ liệu đã chuẩn bị:
{
  "instruction": "Convert the natural language question to a SQL query based on the given database schema.",
  "input": "### SQLite Schema:\n```sql\nCREATE TABLE IF NOT EXISTS \"department\" (\"Department_ID\" int, \"Name\" text, \"Creation\" text, \"Ranking\" int, \"Budget_in_Billions\" real, \"Num_Employees\" real, PRIMARY KEY (\"Department_ID\"));\n\n\nCREATE TABLE IF NOT EXISTS \"head\" (\"head_ID\" int, \"name\" text, \"born_state\" text, \"age\" real, PRIMARY KEY (\"head_ID\"));\n\n\nCREATE TABLE IF NOT EXISTS \"management\" (\"department_ID\" int, \"head_ID\" int, \"temporary_acting\" text, PRIMARY KEY (\"Department_ID\",\n                                                                                                                   \"head_ID\"),\n                                         FO

In [ ]:
# PHẦN 6: TẢI MODEL VÀ TOKENIZER SỬ DỤNG UNSLOTH
from unsloth import FastLanguageModel
import torch

# Chọn model phù hợp
model_name = "unsloth/Qwen2.5-7B"

# Tải model và tokenizer
print("Đang tải model và tokenizer...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Áp dụng LoRA vào model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                # Rank
    lora_alpha=16,       # Alpha scaling
    lora_dropout=0,      # Dropout probability
    bias="none",         # Bias type
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config = None,
)

print(f"Tổng số tham số: {model.num_parameters()}")
print(f"Số tham số được huấn luyện: {model.num_parameters(True)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Đang tải model và tokenizer...
==((====))==  Unsloth 2025.3.9: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/106k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2025.3.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Tổng số tham số: 7655986688
Số tham số được huấn luyện: 40370176


In [ ]:
# PHẦN 7: TOKENIZE DỮ LIỆU
CUTOFF_LEN = 2048  # Độ dài tối đa sau khi tokenize

# Lấy EOS token
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Sử dụng định dạng chat của Qwen2.5
        text = f"<|im_start|>user\n{instruction}\n\n{input}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>{EOS_TOKEN}"
        texts.append(text)

    return {"text": texts}

# Áp dụng formatting
print("Đang chuẩn bị dữ liệu huấn luyện...")
formatted_train = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_dataset.column_names
)

print("Đang chuẩn bị dữ liệu kiểm thử...")
formatted_valid = valid_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=valid_dataset.column_names
)

# Kiểm tra kết quả
print(f"Dữ liệu huấn luyện sau khi định dạng: {formatted_train}")
print(f"Dữ liệu kiểm thử sau khi định dạng: {formatted_valid}")

Đang chuẩn bị dữ liệu huấn luyện...


Map:   0%|          | 0/6654 [00:00<?, ? examples/s]

Đang chuẩn bị dữ liệu kiểm thử...


Map:   0%|          | 0/807 [00:00<?, ? examples/s]

Dữ liệu huấn luyện sau khi định dạng: Dataset({
    features: ['text'],
    num_rows: 6654
})
Dữ liệu kiểm thử sau khi định dạng: Dataset({
    features: ['text'],
    num_rows: 807
})


In [ ]:
# PHẦN 8: THIẾT LẬP THAM SỐ HUẤN LUYỆN
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported  # Thêm hàm kiểm tra hỗ trợ bfloat16

# Thiết lập các tham số huấn luyện
training_args = TrainingArguments(
    output_dir="./qwen2_5_nl2sql",
    # num_train_epochs=3,           # Có thể giữ lại hoặc dùng max_steps
    max_steps=300,                 # Sử dụng max_steps thay vì epochs
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",         # Đã thay evaluation_strategy thành eval_strategy
    logging_steps=1,               # Ghi log thường xuyên
    eval_steps=20,                # Giảm số bước đánh giá
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=2e-4,
    warmup_steps=5,                # Sử dụng số bước cố định thay vì tỷ lệ
    weight_decay=0.01,
    fp16=not is_bfloat16_supported(),  # Động lựa chọn giữa fp16 và bf16
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",            # Sử dụng optimizer 8-bit để tiết kiệm bộ nhớ
    max_grad_norm=0.3,
    save_total_limit=3,
    seed=3407,                     # Thiết lập seed cố định
    report_to="none",       # Theo dõi huấn luyện bằng TensorBoard
)

In [ ]:
# PHẦN 9: HUẤN LUYỆN MODEL
# Sử dụng Unsloth Trainer
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train,
    eval_dataset=formatted_valid,
    args=training_args,
    packing=True,                  # Đóng gói nhiều mẫu vào cùng một chuỗi
    dataset_text_field="text",     # Chỉ định trường chứa văn bản đã định dạng
    max_seq_length=CUTOFF_LEN,     # Độ dài chuỗi đầu vào tối đa
)

Tokenizing to ["text"] (num_proc=12):   0%|          | 0/6654 [00:00<?, ? examples/s]

Packing train dataset (num_proc=12):   0%|          | 0/6654 [00:00<?, ? examples/s]

Tokenizing to ["text"] (num_proc=12):   0%|          | 0/807 [00:00<?, ? examples/s]

Packing eval dataset (num_proc=12):   0%|          | 0/807 [00:00<?, ? examples/s]

In [ ]:
# Bắt đầu huấn luyện
print("Bắt đầu quá trình huấn luyện...")
trainer.train()

Bắt đầu quá trình huấn luyện...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,762 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176/5,063,120,384 (0.80% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,0.318300,0.318698
40,0.234700,0.283716
60,0.216300,0.288668
80,0.174200,0.280150
100,0.181800,0.278706
120,0.145200,0.289060
140,0.110900,0.293030
160,0.171300,0.302638
180,0.194200,0.298572
200,0.119000,0.303578


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Could not locate the best model at ./qwen2_5_nl2sql/checkpoint-100/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


TrainOutput(global_step=300, training_loss=0.18653054611136516, metrics={'train_runtime': 1892.8531, 'train_samples_per_second': 1.268, 'train_steps_per_second': 0.158, 'total_flos': 2.0918732897805926e+17, 'train_loss': 0.18653054611136516})

In [ ]:
# PHẦN 10: LƯU MODEL ĐÃ HUẤN LUYỆN
save_directory = "qwen2_5_nl2sql_finetuned"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Đã lưu model tại: {save_directory}")

Đã lưu model tại: qwen2_5_nl2sql_finetuned


In [ ]:
# Lưu model vào Google Drive
drive_save_directory = os.path.join(DRIVE_OUTPUT_DIR, "qwen2_5_nl2sql_finetuned")
os.makedirs(drive_save_directory, exist_ok=True)

# Sao chép các file model sang Google Drive
import shutil
def copy_directory_to_drive(src, dst):
    """Sao chép thư mục nguồn sang đích và hiển thị tiến trình"""
    if not os.path.exists(dst):
        os.makedirs(dst)

    files = os.listdir(src)
    total_files = len(files)

    print(f"Bắt đầu sao chép {total_files} files từ {src} sang {dst}")

    for i, item in enumerate(files, 1):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)

        if os.path.isdir(s):
            copy_directory_to_drive(s, d)
        else:
            file_size_mb = os.path.getsize(s) / (1024 * 1024)
            print(f"[{i}/{total_files}] Đang sao chép {item} ({file_size_mb:.2f} MB)...")
            shutil.copy2(s, d)

    print(f"Đã sao chép xong thư mục {src}")

# Sao chép model
copy_directory_to_drive(save_directory, drive_save_directory)

# Lưu config và tokenizer riêng
config_file = os.path.join(save_directory, "config.json")
if os.path.exists(config_file):
    shutil.copy2(config_file, os.path.join(DRIVE_OUTPUT_DIR, "config.json"))

# Tạo file README với thông tin về model
with open(os.path.join(DRIVE_OUTPUT_DIR, "README.txt"), "w") as f:
    f.write(f"""# Qwen2.5 Fine-tuned for Text-to-SQL

Base model: {model_name}

## Thông số huấn luyện:
- Learning rate: {training_args.learning_rate}
- Batch size: {training_args.per_device_train_batch_size}
- Gradient accumulation steps: {training_args.gradient_accumulation_steps}
- Total steps: {training_args.max_steps}
- LoRA rank: 16
- LoRA alpha: 32

## Thông tin sử dụng:
Để sử dụng model này, hãy load bằng:

```python
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load tokenizer và model gốc
model_name = "{model_name}"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load adapter LoRA
adapter_path = "đường/dẫn/tới/qwen2_5_nl2sql_finetuned"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_path)
```
""")

print(f"Đã lưu model thành công vào Google Drive tại: {drive_save_directory}")
print(f"Bạn có thể tải xuống model từ Google Drive > {DRIVE_OUTPUT_DIR}")

# Hiển thị đường dẫn tuyệt đối để dễ tìm trên Google Drive
import subprocess
try:
    drive_path = subprocess.check_output("find /content/drive -type d -name 'qwen2_5_nl2sql_finetuned'", shell=True).decode().strip()
    print(f"Đường dẫn đầy đủ trên Drive: {drive_path}")
except:
    print("Không thể tìm thấy đường dẫn tuyệt đối. Vui lòng kiểm tra thủ công.")

# Kiểm tra kích thước thư mục đã lưu
drive_size = subprocess.check_output(f"du -sh '{drive_save_directory}'", shell=True).decode().strip()
print(f"Kích thước model trên Drive: {drive_size}")

Bắt đầu sao chép 10 files từ qwen2_5_nl2sql_finetuned sang /content/drive/MyDrive/Qwen2_5_SQL_Model_3/qwen2_5_nl2sql_finetuned
[1/10] Đang sao chép README.md (0.00 MB)...
[2/10] Đang sao chép vocab.json (2.65 MB)...
[3/10] Đang sao chép tokenizer.json (10.89 MB)...
[4/10] Đang sao chép added_tokens.json (0.00 MB)...
[5/10] Đang sao chép special_tokens_map.json (0.00 MB)...
[6/10] Đang sao chép tokenizer_config.json (0.00 MB)...
[7/10] Đang sao chép adapter_model.safetensors (154.05 MB)...
[8/10] Đang sao chép adapter_config.json (0.00 MB)...
[9/10] Đang sao chép merges.txt (1.59 MB)...
[10/10] Đang sao chép training_args.bin (0.01 MB)...
Đã sao chép xong thư mục qwen2_5_nl2sql_finetuned
Đã lưu model thành công vào Google Drive tại: /content/drive/MyDrive/Qwen2_5_SQL_Model_3/qwen2_5_nl2sql_finetuned
Bạn có thể tải xuống model từ Google Drive > /content/drive/MyDrive/Qwen2_5_SQL_Model_3
Đường dẫn đầy đủ trên Drive: /content/drive/MyDrive/Qwen2_5_SQL_Model/qwen2_5_nl2sql_finetuned
/conten

In [ ]:
if True: model.save_pretrained_gguf("model", tokenizer,)

Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 7.5G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 61.85 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 28/28 [00:00<00:00, 52.55it/s]


Unsloth: Saving tokenizer... Done.
Done.


Unsloth: Converting qwen2 model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at model into q8_0 GGUF format.
The output location will be /content/model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> Q8_0, shape = {35

In [ ]:
import os
drive_destination = "/content/drive/MyDrive/Model"
os.makedirs(drive_destination, exist_ok=True)

# Copy toàn bộ thư mục với hiển thị tiến trình
import shutil
import os

def copy_with_progress(src_folder, dst_folder):
    # Tạo thư mục đích nếu chưa tồn tại
    if not os.path.exists(dst_folder):
        os.makedirs(dst_folder)

    # Lấy danh sách tất cả các file trong thư mục nguồn
    files = os.listdir(src_folder)
    total_files = len(files)

    print(f"Bắt đầu sao chép {total_files} files từ {src_folder} sang {dst_folder}")

    # Copy từng file với hiển thị tiến trình
    for i, file_name in enumerate(files, 1):
        src_path = os.path.join(src_folder, file_name)
        dst_path = os.path.join(dst_folder, file_name)

        # Nếu là thư mục con
        if os.path.isdir(src_path):
            copy_with_progress(src_path, dst_path)
        else:
            # Hiển thị tiến trình
            file_size = os.path.getsize(src_path) / (1024 * 1024)  # Kích thước MB
            print(f"[{i}/{total_files}] Đang sao chép: {file_name} ({file_size:.2f} MB)")

            # Copy file
            shutil.copy2(src_path, dst_path)

    print(f"Đã hoàn tất sao chép thư mục {src_folder}")

# Thực hiện sao chép
source_folder = "/content/model"  # Đường dẫn đến thư mục model của bạn
copy_with_progress(source_folder, drive_destination)

print(f"\nĐã sao chép thành công thư mục model vào Google Drive tại: {drive_destination}")

Bắt đầu sao chép 14 files từ /content/model sang /content/drive/MyDrive/Model
[1/14] Đang sao chép: generation_config.json (0.00 MB)
[2/14] Đang sao chép: model-00004-of-00004.safetensors (1039.50 MB)
[3/14] Đang sao chép: vocab.json (2.65 MB)
[4/14] Đang sao chép: tokenizer.json (10.89 MB)
[5/14] Đang sao chép: model.safetensors.index.json (0.03 MB)
[6/14] Đang sao chép: added_tokens.json (0.00 MB)
[7/14] Đang sao chép: model-00002-of-00004.safetensors (4704.24 MB)
[8/14] Đang sao chép: special_tokens_map.json (0.00 MB)
[9/14] Đang sao chép: unsloth.Q8_0.gguf (7723.35 MB)
[10/14] Đang sao chép: tokenizer_config.json (0.00 MB)
[11/14] Đang sao chép: model-00003-of-00004.safetensors (4130.23 MB)
[12/14] Đang sao chép: merges.txt (1.59 MB)
[13/14] Đang sao chép: model-00001-of-00004.safetensors (4651.70 MB)
[14/14] Đang sao chép: config.json (0.00 MB)
Đã hoàn tất sao chép thư mục /content/model

Đã sao chép thành công thư mục model vào Google Drive tại: /content/drive/MyDrive/Model


In [ ]:
# PHẦN 11: KIỂM TRA VỚI MẪU TEST
def generate_sql(model, tokenizer, schema_content, question):
    """Tạo câu truy vấn SQL từ schema và câu hỏi"""
    # Sử dụng hàm make_prompt để nhất quán giữa training và inference
    prompt = make_prompt(question, schema_content)

    # Tạo prompt theo định dạng Qwen2.5
    prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            num_return_sequences=1,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Trích xuất phần SQL từ phản hồi
    import re
    sql_match = re.search(r"```sql\n(.*?)\n```", response, re.DOTALL)
    if sql_match:
        return sql_match.group(1).strip()
    else:
        return response  # Trả về toàn bộ nếu không tìm thấy định dạng SQL cụ thể

# Tải lại model đã fine-tune để kiểm tra
checkpoint_path = save_directory  # Hoặc có thể chọn checkpoint tốt nhất từ eval
print(f"Đang tải model từ {checkpoint_path} để kiểm tra...")

# Tải model đã huấn luyện để kiểm tra với Unsloth
from peft import PeftModel
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,  # model gốc
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True
)
finetuned_model = PeftModel.from_pretrained(base_model, checkpoint_path)
finetuned_tokenizer = base_tokenizer

# Lấy một mẫu từ tập validation để kiểm tra
test_sample = valid_df.iloc[0]
db_id = test_sample['db_id']
test_question = test_sample['question']
test_schema = read_schema_content(db_id)

print("Câu hỏi mẫu:", test_question)
print("Câu truy vấn SQL thực tế:", test_sample['query'])
print("Câu truy vấn SQL được tạo bởi model:")
print(generate_sql(finetuned_model, finetuned_tokenizer, test_schema, test_question))

Đang tải model từ qwen2_5_nl2sql_finetuned để kiểm tra...
==((====))==  Unsloth 2025.3.9: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Câu hỏi mẫu: How many singers do we have?
Câu truy vấn SQL thực tế: SELECT count(*) FROM singer
Câu truy vấn SQL được tạo bởi model:
CREATE TABLE "stadium" ("Stadium_ID" int, "Location" text, "Name" text, "Capacity" int, "Highest" int, "Lowest" int, "Average" int, PRIMARY KEY ("Stadium_ID"));


CREATE TABLE "singer" ("Singer_ID" int, "Name" text, "Country" text, "Song_Name" text, "Song_release_year" text, "Age" int, "Is_male" bool,
                                                                                                                            PRIMARY KEY ("Singer_ID"));


CREATE TABLE "concert" ("concert_ID" int, "concert_Name" text, "Theme" text, "Stadium_ID" text, "Year" text, PRIMARY KEY ("concert_ID"),
                        FOREIGN KEY ("Stadium_ID") REFERENCES "stadium"("Stadium_ID"));


CREATE TABLE "singer_in_concert" ("concert_ID" int, "Singer_ID" text, PRIMARY KEY ("concert_ID",
                                                                                   "Si

In [ ]:
# PHẦN 12: KIỂM TRA MODEL SAU KHI HUẤN LUYỆN
print("\nKiểm tra model sau khi huấn luyện:")

# Tạo hàm để kiểm tra model với một câu hỏi
def test_model(question, db_id):
    # Đọc schema
    schema_content = read_schema_content(db_id)
    if schema_content == "Schema không tồn tại" or schema_content == "Lỗi khi đọc schema":
        return "Không tìm thấy schema cho database này"

    # Tạo prompt
    prompt = make_prompt(question, schema_content)

    # Format prompt theo chuẩn Qwen2.5
    chat_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

    # Tạo đầu ra từ model
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode đầu ra
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    return response

# Thử nghiệm với một vài câu hỏi từ tập valid
try:
    for i in range(min(20, len(valid_df))):
        question = valid_df.iloc[i]["question"]
        db_id = valid_df.iloc[i]["db_id"]
        expected_query = valid_df.iloc[i]["query"]

        print(f"\nCâu hỏi: {question}")
        print(f"Database: {db_id}")
        print(f"Kết quả mong đợi: {expected_query}")

        response = test_model(question, db_id)
        print(f"Kết quả từ model: {response}")
        print("-" * 80)
except Exception as e:
    print(f"Lỗi khi kiểm tra model: {e}")


Kiểm tra model sau khi huấn luyện:
Lỗi khi kiểm tra model: name 'valid_df' is not defined


In [ ]:
# PHẦN 13: KIỂM TRA MODEL
print("\nKiểm tra model sau khi huấn luyện:")

# Tạo hàm để kiểm tra model với một câu hỏi
def test_model(question, schema_content):
    """Kiểm tra model với câu hỏi và schema được cung cấp"""
    # Tạo prompt
    prompt = make_prompt(question, schema_content)

    # Format prompt theo chuẩn Qwen2.5
    chat_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

    # Tạo đầu ra từ model
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False
    )

    # Decode đầu ra
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    return response

# Hàm kiểm tra với dữ liệu từ tập validation
def test_with_validation_data(num_samples=3):
    print("\n===== KIỂM TRA VỚI DỮ LIỆU TỪ TẬP VALIDATION =====")
    try:
        for i in range(min(num_samples, len(valid_df))):
            question = valid_df.iloc[i]["question"]
            db_id = valid_df.iloc[i]["db_id"]
            expected_query = valid_df.iloc[i]["query"]

            # Đọc schema từ db_id
            schema_content = read_schema_content(db_id)
            if schema_content == "Schema không tồn tại" or schema_content == "Lỗi khi đọc schema":
                print(f"Không tìm thấy schema cho database {db_id}")
                continue

            print(f"\nCâu hỏi: {question}")
            print(f"Database: {db_id}")
            print(f"Kết quả mong đợi: {expected_query}")

            response = test_model(question, schema_content)
            print(f"Kết quả từ model: {response}")
            print("-" * 80)
    except Exception as e:
        print(f"Lỗi khi kiểm tra model với dữ liệu validation: {e}")

# Hàm kiểm tra với đầu vào tùy chỉnh
def test_with_custom_input():
    print("\n===== KIỂM TRA VỚI ĐẦU VÀO TÙY CHỈNH =====")
    try:
        while True:
            # Nhập schema trực tiếp
            print("\nNhập schema database (nhập 'done' để kết thúc):")
            schema_lines = []
            while True:
                line = input()
                if line.lower() == 'done':
                    break
                schema_lines.append(line)

            if not schema_lines:
                print("Kết thúc kiểm tra.")
                break

            schema_content = "\n".join(schema_lines)

            # Nhập câu hỏi
            question = input("\nNhập câu hỏi của bạn: ")

            # Thực hiện kiểm tra
            print("\nĐang tạo SQL...")
            response = test_model(question, schema_content)
            print(f"\nKết quả từ model:")
            print(response)
            print("-" * 80)

            # Hỏi người dùng có muốn tiếp tục không
            cont = input("\nTiếp tục kiểm tra? (y/n): ")
            if cont.lower() != 'y':
                print("Kết thúc kiểm tra.")
                break
    except Exception as e:
        print(f"Lỗi khi kiểm tra model với đầu vào tùy chỉnh: {e}")

# Menu lựa chọn
print("\nLựa chọn chế độ kiểm tra:")
print("1. Kiểm tra với dữ liệu từ tập validation")
print("2. Kiểm tra với đầu vào tùy chỉnh")
choice = input("Nhập lựa chọn của bạn (1 hoặc 2): ")

if choice == "1":
    num_samples = int(input("Nhập số lượng mẫu muốn kiểm tra: "))
    test_with_validation_data(num_samples)
elif choice == "2":
    test_with_custom_input()
else:
    print("Lựa chọn không hợp lệ!")


Kiểm tra model sau khi huấn luyện:

Lựa chọn chế độ kiểm tra:
1. Kiểm tra với dữ liệu từ tập validation
2. Kiểm tra với đầu vào tùy chỉnh
Nhập lựa chọn của bạn (1 hoặc 2): 2

===== KIỂM TRA VỚI ĐẦU VÀO TÙY CHỈNH =====

Nhập schema database (nhập 'done' để kết thúc):
CREATE TABLE students (   id INTEGER PRIMARY KEY,   name TEXT NOT NULL,   age INTEGER,   major TEXT,   gpa REAL );  CREATE TABLE courses (   course_id TEXT PRIMARY KEY,   title TEXT NOT NULL,   credits INTEGER,   department TEXT ); done
Liệt kê tất cả sinh viên chuyên ngành Computer Science
done

Nhập câu hỏi của bạn: Liệt kê tất cả sinh viên chuyên ngành Computer Science

Đang tạo SQL...

Kết quả từ model:
```sql
SELECT *
FROM students
WHERE major = "Computer Science"
``` backpage
--------------------------------------------------------------------------------

Tiếp tục kiểm tra? (y/n): n
Kết thúc kiểm tra.


In [ ]:
# PHẦN 14: ĐÁNH GIÁ MÔ HÌNH TEXT-TO-SQL
import re
import numpy as np
import sqlparse
from sklearn.metrics import f1_score, precision_score, recall_score
import sqlite3
import pandas as pd
import os

def normalize_sql(sql):
    # Chuẩn hóa câu lệnh SQL để giảm thiểu sự khác biệt cú pháp
    sql = sql.lower()
    # Loại bỏ dấu chấm phẩy cuối câu
    sql = sql.rstrip(';').strip()
    # Loại bỏ khoảng trắng thừa
    sql = re.sub(r'\s+', ' ', sql)
    # Chuẩn hóa từ khóa AS trong bí danh
    sql = re.sub(r' as ', ' ', sql)
    # Chuẩn hóa quotes
    sql = re.sub(r'[\'"`]', '"', sql)
    return sql

def exact_match_accuracy(pred_queries, gold_queries):
    """Tính Exact Match Accuracy"""
    correct = 0
    for pred, gold in zip(pred_queries, gold_queries):
        if normalize_sql(pred) == normalize_sql(gold):
            correct += 1
    return correct / len(gold_queries) if len(gold_queries) > 0 else 0

def component_matching(pred_query, gold_query):
    """Phân tích và so sánh các thành phần của câu truy vấn SQL"""
    # Chuẩn hóa và parse SQL
    pred_norm = normalize_sql(pred_query)
    gold_norm = normalize_sql(gold_query)

    # Phân tích thành phần
    components = {
        'select': extract_component(r'select(.*?)(?:from|where|group|order|limit|$)', pred_norm, gold_norm),
        'from': extract_component(r'from(.*?)(?:where|group|order|limit|$)', pred_norm, gold_norm),
        'where': extract_component(r'where(.*?)(?:group|order|limit|$)', pred_norm, gold_norm),
        'group_by': extract_component(r'group by(.*?)(?:having|order|limit|$)', pred_norm, gold_norm),
        'order_by': extract_component(r'order by(.*?)(?:limit|$)', pred_norm, gold_norm),
    }

    # Tính điểm cho từng thành phần
    component_scores = {}
    for component, (pred_comp, gold_comp) in components.items():
        if not pred_comp and not gold_comp:  # Cả hai đều không có thành phần này
            component_scores[component] = 1.0
        elif not pred_comp or not gold_comp:  # Một trong hai không có
            component_scores[component] = 0.0
        else:
            # Tính độ tương đồng giữa các thành phần
            pred_tokens = set(re.findall(r'\b\w+\b', pred_comp))
            gold_tokens = set(re.findall(r'\b\w+\b', gold_comp))

            if len(gold_tokens) == 0:
                component_scores[component] = 1.0 if len(pred_tokens) == 0 else 0.0
            else:
                precision = len(pred_tokens.intersection(gold_tokens)) / len(pred_tokens) if pred_tokens else 0
                recall = len(pred_tokens.intersection(gold_tokens)) / len(gold_tokens) if gold_tokens else 0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                component_scores[component] = f1

    # Tính điểm tổng hợp
    avg_score = sum(component_scores.values()) / len(component_scores)
    return component_scores, avg_score

def extract_component(pattern, pred, gold):
    """Trích xuất thành phần từ câu SQL dựa trên pattern regex"""
    pred_match = re.search(pattern, pred)
    gold_match = re.search(pattern, gold)
    return (pred_match.group(1).strip() if pred_match else "",
            gold_match.group(1).strip() if gold_match else "")

def execution_accuracy(pred_queries, gold_queries, db_paths):
    """Tính Execution Accuracy bằng cách thực thi SQL và so sánh kết quả"""
    correct = 0
    skipped = 0

    for pred, gold, db_path in zip(pred_queries, gold_queries, db_paths):
        # Kiểm tra xem database có tồn tại không
        if not os.path.exists(db_path):
            skipped += 1
            continue

        try:
            # Kết nối database
            conn = sqlite3.connect(db_path)

            # Thực thi câu truy vấn gold và lấy kết quả
            gold_result = pd.read_sql_query(gold, conn)

            # Thực thi câu truy vấn được dự đoán và lấy kết quả
            pred_result = pd.read_sql_query(pred, conn)

            # So sánh kết quả (bỏ qua thứ tự hàng)
            gold_result = gold_result.sort_values(by=gold_result.columns.tolist()).reset_index(drop=True)
            pred_result = pred_result.sort_values(by=pred_result.columns.tolist()).reset_index(drop=True)

            # So sánh DataFrame
            if gold_result.equals(pred_result):
                correct += 1

            conn.close()
        except Exception as e:
            print(f"Lỗi khi thực thi SQL: {e}")
            skipped += 1

    # Trả về accuracy dựa trên số lượng câu truy vấn không bị bỏ qua
    valid_count = len(gold_queries) - skipped
    return correct / valid_count if valid_count > 0 else 0, skipped

def evaluate_model(model, tokenizer, valid_df, dbid2schema_path, num_samples=None):
    """Đánh giá toàn diện model Text-to-SQL"""
    if num_samples is None or num_samples > len(valid_df):
        num_samples = len(valid_df)

    # Lấy mẫu từ tập validation
    eval_df = valid_df.sample(num_samples) if num_samples < len(valid_df) else valid_df

    print(f"Đánh giá model trên {num_samples} mẫu từ tập validation...")

    pred_queries = []
    gold_queries = []
    db_paths = []
    component_scores = []

    # Tạo các dự đoán
    for idx, row in eval_df.iterrows():
        question = row["question"]
        db_id = row["db_id"]
        gold_query = row["query"]

        # Đọc schema
        schema_content = read_schema_content(db_id)
        if schema_content == "Schema không tồn tại" or schema_content == "Lỗi khi đọc schema":
            continue

        # Tạo prompt và dự đoán SQL
        prompt = make_prompt(question, schema_content)
        chat_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

        # Tokenize và tạo dự đoán
        inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False
        )

        # Giải mã kết quả
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        # Trích xuất SQL từ phản hồi
        sql_match = re.search(r"```sql\n(.*?)\n```", response, re.DOTALL)
        if sql_match:
            pred_query = sql_match.group(1).strip()
        else:
            pred_query = response.strip()

        # Lưu lại kết quả
        pred_queries.append(pred_query)
        gold_queries.append(gold_query)

        # Lưu đường dẫn đến database để thực thi sau này
        db_path = os.path.join("database", db_id, f"{db_id}.sqlite")
        db_paths.append(db_path)

        # Tính component matching
        comp_score, avg_comp_score = component_matching(pred_query, gold_query)
        component_scores.append(avg_comp_score)

        # In tiến trình
        if (idx + 1) % 10 == 0:
            print(f"Đã xử lý {idx + 1}/{num_samples} mẫu...")

    # Tính các metrics
    em_accuracy = exact_match_accuracy(pred_queries, gold_queries)
    avg_component_score = np.mean(component_scores)

    # Thử tính execution accuracy nếu có thể
    try:
        exec_accuracy, skipped = execution_accuracy(pred_queries, gold_queries, db_paths)
        exec_available = True
    except Exception as e:
        print(f"Không thể tính Execution Accuracy: {e}")
        exec_accuracy = None
        skipped = None
        exec_available = False

    # In kết quả
    print("\n===== KẾT QUẢ ĐÁNH GIÁ =====")
    print(f"Số lượng mẫu đánh giá: {len(pred_queries)}")
    print(f"Exact Match Accuracy: {em_accuracy:.4f}")
    print(f"Component Matching Score: {avg_component_score:.4f}")

    if exec_available:
        print(f"Execution Accuracy: {exec_accuracy:.4f} (đã bỏ qua {skipped} mẫu)")

    # Lưu kết quả chi tiết
    results_df = pd.DataFrame({
        'question': eval_df['question'].values[:len(pred_queries)],
        'db_id': eval_df['db_id'].values[:len(pred_queries)],
        'gold_query': gold_queries,
        'pred_query': pred_queries,
        'exact_match': [normalize_sql(pred) == normalize_sql(gold) for pred, gold in zip(pred_queries, gold_queries)],
        'component_score': component_scores
    })

    results_df.to_csv("evaluation_results.csv", index=False)
    print("Đã lưu kết quả chi tiết vào file evaluation_results.csv")

    return {
        'exact_match_accuracy': em_accuracy,
        'component_matching_score': avg_component_score,
        'execution_accuracy': exec_accuracy if exec_available else None,
        'results_df': results_df
    }

# Sử dụng hàm đánh giá
try:
    eval_results = evaluate_model(model, tokenizer, valid_df, dbid2schema_path, num_samples=100)
    print("\nSummary:")
    print(f"Exact Match Accuracy: {eval_results['exact_match_accuracy']:.4f}")
    print(f"Component Matching Score: {eval_results['component_matching_score']:.4f}")
    if eval_results['execution_accuracy'] is not None:
        print(f"Execution Accuracy: {eval_results['execution_accuracy']:.4f}")
except Exception as e:
    print(f"Lỗi khi đánh giá model: {e}")

Đánh giá model trên 100 mẫu từ tập validation...
Đã xử lý 710/100 mẫu...
Đã xử lý 150/100 mẫu...
Đã xử lý 510/100 mẫu...
Đã xử lý 750/100 mẫu...
Đã xử lý 500/100 mẫu...
Đã xử lý 190/100 mẫu...
Đã xử lý 400/100 mẫu...
Đã xử lý 590/100 mẫu...
Đã xử lý 470/100 mẫu...
Lỗi khi thực thi SQL: Could not decode to UTF-8 column 'last_name' with text 'Treyes Albarrac��N'
Lỗi khi thực thi SQL: Execution failed on sql 'SELECT T1.first_name,
       T1.last_name,
       T2.size_description
FROM owners AS T1
JOIN dogs AS T2 ON T1.owner_id = T2.owner_id': no such column: T2.size_description
Lỗi khi thực thi SQL: Execution failed on sql 'SELECT template_id
FROM templates
WHERE template_type_description = 'Presentation'': no such column: template_type_description
Lỗi khi thực thi SQL: Execution failed on sql 'SELECT T1.Name,
       T1.Song_Name
FROM singer AS T1
JOIN song AS T2 ON T1.Singer_ID = T2.Singer_ID
ORDER BY T1.Age
LIMIT 1': no such table: song
Lỗi khi thực thi SQL: Execution failed on sql 'SELE